Imports and installs


In [3]:
%pip install scikit-learn

  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
   ---------------------------------------- 0.0/8.1 MB ? eta -:--:--
   ----------------------------- ---------- 6.0/8.1 MB 46.1 MB/s eta 0:00:01
   ---------------------------------------- 8.1/8.1 MB 45.5 MB/s  0:00:00
Using cached threadpoolctl-3.6.0-py3-none-any.whl (18 kB)

   ------------- -------------------------- 1/3 [joblib]
   ------------- -------------------------- 1/3 [joblib]
   -------------------------- ------------- 2/3 [scikit-learn]
   -------------------------- ------------- 2/3 [scikit-learn]
   -------------------------- ------------- 2/3 [scikit-learn]
   -------------------------- ------------- 2/3 [scikit-learn]
   -------------------------- ------------- 2/3 [scikit-learn]
   -------------------------- ------------- 2/3 [scikit-learn]
   -------------------------- ------------- 2/3 [scikit-learn]
   -------------------------- ------------- 2/3 [scikit-learn]
   -------------------------- ----

In [19]:
import os
import gc
import re
import json
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, average_precision_score

In [5]:
DATA_PATH = r"D:/build_master_outputs/master_dataset_final_ml_v3.parquet"

df = pd.read_parquet(DATA_PATH)

print(df.shape)
print(df.head())
print(df.dtypes.sort_index())

(211794, 64)
   EID    MONTH  PEAKID  is_sim_only   PR    C  PROFIT  TARGET  \
0    2  2020-01       0            0  0.0  0.0     0.0       0   
1    2  2020-01       1            0  0.0  0.0     0.0       0   
2    3  2020-01       0            0  0.0  0.0     0.0       0   
3    3  2020-01       1            0  0.0  0.0     0.0       0   
4   10  2020-01       0            0  0.0  0.0     0.0       0   

   pr_partial_current DECISION_MONTH  ...  psm_hydro_abs_mean  \
0                 0.0        2019-12  ...            7.404401   
1                 0.0        2019-12  ...            7.892746   
2                 0.0        2019-12  ...           47.846898   
3                 0.0        2019-12  ...           16.726493   
4                 0.0        2019-12  ...          115.069268   

   psm_nonrenew_abs_mean  psm_external_abs_mean  psm_abs_s1_mean  \
0              12.413350               1.749670         0.000000   
1              13.707072               0.908503         0.00000

Timeline Preparation and cleaning


In [20]:
# Force MONTH to string YYYY-MM
df["MONTH"] = df["MONTH"].astype(str)

# Basic target safety
assert "TARGET" in df.columns, "TARGET column not found."

# Sort months
all_months = sorted(df["MONTH"].dropna().unique().tolist())

print("Nb months:", len(all_months))
print("First 10 months:", all_months[:10])
print("Last 10 months:", all_months[-10:])

print("\nTarget distribution:")
print(df["TARGET"].value_counts(dropna=False))
print(df["TARGET"].value_counts(normalize=True, dropna=False))

Nb months: 48
First 10 months: ['2020-01', '2020-02', '2020-03', '2020-04', '2020-05', '2020-06', '2020-07', '2020-08', '2020-09', '2020-10']
Last 10 months: ['2023-03', '2023-04', '2023-05', '2023-06', '2023-07', '2023-08', '2023-09', '2023-10', '2023-11', '2023-12']

Target distribution:
TARGET
0    196388
1     15406
Name: count, dtype: int64
TARGET
0    0.92726
1    0.07274
Name: proportion, dtype: float64


Sanity Checks

In [21]:
key_cols = [c for c in ["EID", "MONTH", "PEAKID"] if c in df.columns]
print("Key cols:", key_cols)

if len(key_cols) == 3:
    n_dupes = df.duplicated(subset=key_cols).sum()
    print("Duplicate key rows:", n_dupes)
else:
    print("Warning: one or more key columns missing.")

print("\nTop missingness:")
df.isna().mean().sort_values(ascending=False).head(30)

Key cols: ['EID', 'MONTH', 'PEAKID']
Duplicate key rows: 0

Top missingness:


EID                     0.0
MONTH                   0.0
PEAKID                  0.0
is_sim_only             0.0
PR                      0.0
C                       0.0
PROFIT                  0.0
TARGET                  0.0
pr_partial_current      0.0
DECISION_MONTH          0.0
has_pr_history          0.0
has_profit_history      0.0
has_cost_history        0.0
pr_lag1                 0.0
pr_lag2                 0.0
pr_lag3                 0.0
c_lag1                  0.0
profit_lag1             0.0
profit_lag2             0.0
target_lag1             0.0
pr_rolling3_mean        0.0
profit_rolling3_mean    0.0
profitable_count_3m     0.0
psd_nonzero_count       0.0
psd_abs_nonzero_mean    0.0
psd_abs_nonzero_std     0.0
psd_abs_sum             0.0
psd_signed_mean         0.0
psd_abs_max             0.0
activation_mean         0.0
dtype: float64

Cell 5 — Anti-leakage feature policy


   

In [23]:
TARGET_COL = "TARGET"
MONTH_COL = "MONTH"

ID_COLS = [c for c in ["EID"] if c in df.columns]
BASE_DROP_COLS = [TARGET_COL, MONTH_COL] + ID_COLS

# Colonnes explicitement dangereuses si présentes
EXPLICIT_LEAKAGE_COLS = [
    "C",
    "PR",
    "PR_signed",
    "PROFIT",
]

# Patterns suspects à exclure par prudence
# On reste conservateur pour le premier modèle
LEAKAGE_PATTERNS = [
    r"^target$",
    r"^c$",
    r"^pr$",
    r"^pr_signed$",
    r"^profit$",
    r"actual",
    r"realized",
    r"label",
]

def matches_any_pattern(col_name, patterns):
    col_low = col_name.lower()
    return any(re.search(p, col_low) for p in patterns)

pattern_drop_cols = [c for c in df.columns if matches_any_pattern(c, LEAKAGE_PATTERNS)]

manual_drop_cols = sorted(set(
    [c for c in BASE_DROP_COLS + EXPLICIT_LEAKAGE_COLS + pattern_drop_cols if c in df.columns]
))

print("Dropped columns for safety:")
print(manual_drop_cols)
print("Nb dropped:", len(manual_drop_cols))

Dropped columns for safety:
['C', 'EID', 'MONTH', 'PR', 'PROFIT', 'TARGET']
Nb dropped: 6


Cell 6 — Build candidate feature list

In [24]:
candidate_feature_cols = [c for c in df.columns if c not in manual_drop_cols]

# Keep only numeric features for the first baseline
feature_cols = df[candidate_feature_cols].select_dtypes(include=[np.number]).columns.tolist()

print("Numeric candidate features:", len(feature_cols))
print(feature_cols[:100])

Numeric candidate features: 56
['PEAKID', 'is_sim_only', 'pr_partial_current', 'has_pr_history', 'has_profit_history', 'has_cost_history', 'pr_lag1', 'pr_lag2', 'pr_lag3', 'c_lag1', 'profit_lag1', 'profit_lag2', 'target_lag1', 'pr_rolling3_mean', 'profit_rolling3_mean', 'profitable_count_3m', 'psd_nonzero_count', 'psd_abs_nonzero_mean', 'psd_abs_nonzero_std', 'psd_abs_sum', 'psd_signed_mean', 'psd_abs_max', 'activation_mean', 'activation_max', 'activation_nonzero_count', 'wind_abs_mean', 'solar_abs_mean', 'hydro_abs_mean', 'nonrenew_abs_mean', 'external_abs_mean', 'load_abs_mean', 'transoutage_abs_mean', 'psd_abs_s1_mean', 'psd_abs_s23_mean', 'psd_scenario_spread', 'psd_abs_early', 'psd_abs_late', 'psm_nonzero_count', 'psm_abs_nonzero_mean', 'psm_abs_nonzero_std', 'psm_abs_sum', 'psm_signed_mean', 'psm_abs_max', 'psm_activation_mean', 'psm_activation_max', 'psm_wind_abs_mean', 'psm_solar_abs_mean', 'psm_hydro_abs_mean', 'psm_nonrenew_abs_mean', 'psm_external_abs_mean', 'psm_abs_s1_mean

Walk-forward split generator

In [25]:
def make_walkforward_splits(months, min_train_months=6, valid_window=1):
    """
    Expanding-window walk-forward CV.

    Example:
      months = [m1,m2,m3,m4,m5,m6,m7]
      min_train_months=4, valid_window=1

      Fold 1: train m1..m4, valid m5
      Fold 2: train m1..m5, valid m6
      Fold 3: train m1..m6, valid m7
    """
    splits = []

    for i in range(min_train_months, len(months) - valid_window + 1):
        train_months = months[:i]
        valid_months = months[i:i + valid_window]
        splits.append((train_months, valid_months))

    return splits

splits = make_walkforward_splits(all_months, min_train_months=6, valid_window=1)

print("Nb folds:", len(splits))
for i, (tr, va) in enumerate(splits[:5], 1):
    print(f"Fold {i}: train {tr[0]}->{tr[-1]} | valid {va[0]}->{va[-1]}")

Nb folds: 42
Fold 1: train 2020-01->2020-06 | valid 2020-07->2020-07
Fold 2: train 2020-01->2020-07 | valid 2020-08->2020-08
Fold 3: train 2020-01->2020-08 | valid 2020-09->2020-09
Fold 4: train 2020-01->2020-09 | valid 2020-10->2020-10
Fold 5: train 2020-01->2020-10 | valid 2020-11->2020-11


Metrics helper

In [26]:
def topk_hit_rate(y_true, y_score, k=100):
    tmp = pd.DataFrame({
        "y_true": np.asarray(y_true),
        "y_score": np.asarray(y_score)
    }).sort_values("y_score", ascending=False)

    topk = tmp.head(k)

    if len(topk) == 0:
        return np.nan, 0

    return float(topk["y_true"].mean()), int(topk["y_true"].sum())

Main walk-forward CV function

In [27]:
def run_walkforward_cv(
    df,
    feature_cols,
    target_col="TARGET",
    month_col="MONTH",
    min_train_months=6,
    valid_window=1,
    topk_list=(50, 100, 250, 500),
    rf_params=None
):
    if rf_params is None:
        rf_params = {
            "n_estimators": 300,
            "max_depth": 8,
            "min_samples_leaf": 20,
            "class_weight": "balanced",
            "random_state": 42,
            "n_jobs": -1
        }

    months = sorted(df[month_col].dropna().astype(str).unique().tolist())
    splits = make_walkforward_splits(
        months=months,
        min_train_months=min_train_months,
        valid_window=valid_window
    )

    cv_rows = []
    oof_parts = []
    feat_imp_parts = []

    print(f"Total folds: {len(splits)}")

    for fold_id, (train_months, valid_months) in enumerate(splits, start=1):
        train_mask = df[month_col].isin(train_months)
        valid_mask = df[month_col].isin(valid_months)

        train_df = df.loc[train_mask].copy()
        valid_df = df.loc[valid_mask].copy()

        X_train = train_df[feature_cols].fillna(0)
        y_train = train_df[target_col].astype(int)

        X_valid = valid_df[feature_cols].fillna(0)
        y_valid = valid_df[target_col].astype(int)

        # Safety checks
        assert set(train_months).isdisjoint(set(valid_months)), "Train/valid month overlap detected."
        assert train_df[month_col].max() < valid_df[month_col].min(), "Temporal ordering broken."

        model = RandomForestClassifier(**rf_params)
        model.fit(X_train, y_train)

        valid_score = model.predict_proba(X_valid)[:, 1]

        row = {
            "fold": fold_id,
            "train_start": train_months[0],
            "train_end": train_months[-1],
            "valid_start": valid_months[0],
            "valid_end": valid_months[-1],
            "n_train": len(train_df),
            "n_valid": len(valid_df),
            "train_pos_rate": float(y_train.mean()),
            "valid_pos_rate": float(y_valid.mean()),
            "roc_auc": roc_auc_score(y_valid, valid_score) if y_valid.nunique() > 1 else np.nan,
            "pr_auc": average_precision_score(y_valid, valid_score) if y_valid.nunique() > 1 else np.nan,
        }

        for k in topk_list:
            hit_rate, n_pos = topk_hit_rate(y_valid, valid_score, k=k)
            row[f"top{k}_hit_rate"] = hit_rate
            row[f"top{k}_positives"] = n_pos

        cv_rows.append(row)

        # OOF predictions
        oof_fold = valid_df.copy()
        oof_fold["cv_fold"] = fold_id
        oof_fold["score"] = valid_score
        oof_parts.append(oof_fold)

        # Feature importance per fold
        feat_imp_fold = pd.DataFrame({
            "feature": feature_cols,
            "importance": model.feature_importances_,
            "fold": fold_id
        })
        feat_imp_parts.append(feat_imp_fold)

        print(
            f"Fold {fold_id:02d} | "
            f"train {train_months[0]} -> {train_months[-1]} | "
            f"valid {valid_months[0]} -> {valid_months[-1]} | "
            f"ROC AUC={row['roc_auc']:.4f} | "
            f"PR AUC={row['pr_auc']:.4f}"
        )

        del train_df, valid_df, X_train, X_valid, y_train, y_valid, model, valid_score, oof_fold, feat_imp_fold
        gc.collect()

    cv_results_df = pd.DataFrame(cv_rows)
    oof_df = pd.concat(oof_parts, ignore_index=True)
    feat_imp_df = pd.concat(feat_imp_parts, ignore_index=True)

    return cv_results_df, oof_df, feat_imp_df

Run the CV

In [28]:
rf_params = {
    "n_estimators": 300,
    "max_depth": 8,
    "min_samples_leaf": 20,
    "class_weight": "balanced",
    "random_state": 42,
    "n_jobs": -1
}

cv_results_df, oof_df, feat_imp_df = run_walkforward_cv(
    df=df,
    feature_cols=feature_cols,
    target_col="TARGET",
    month_col="MONTH",
    min_train_months=6,
    valid_window=1,
    topk_list=(50, 100, 250, 500),
    rf_params=rf_params
)

Total folds: 42
Fold 01 | train 2020-01 -> 2020-06 | valid 2020-07 -> 2020-07 | ROC AUC=0.8178 | PR AUC=0.3638
Fold 02 | train 2020-01 -> 2020-07 | valid 2020-08 -> 2020-08 | ROC AUC=0.8243 | PR AUC=0.4187
Fold 03 | train 2020-01 -> 2020-08 | valid 2020-09 -> 2020-09 | ROC AUC=0.7939 | PR AUC=0.4827
Fold 04 | train 2020-01 -> 2020-09 | valid 2020-10 -> 2020-10 | ROC AUC=0.7943 | PR AUC=0.4229
Fold 05 | train 2020-01 -> 2020-10 | valid 2020-11 -> 2020-11 | ROC AUC=0.7955 | PR AUC=0.4474
Fold 06 | train 2020-01 -> 2020-11 | valid 2020-12 -> 2020-12 | ROC AUC=0.8024 | PR AUC=0.4030
Fold 07 | train 2020-01 -> 2020-12 | valid 2021-01 -> 2021-01 | ROC AUC=0.8300 | PR AUC=0.4418
Fold 08 | train 2020-01 -> 2021-01 | valid 2021-02 -> 2021-02 | ROC AUC=0.8434 | PR AUC=0.4128
Fold 09 | train 2020-01 -> 2021-02 | valid 2021-03 -> 2021-03 | ROC AUC=0.8804 | PR AUC=0.4698
Fold 10 | train 2020-01 -> 2021-03 | valid 2021-04 -> 2021-04 | ROC AUC=0.8179 | PR AUC=0.4207
Fold 11 | train 2020-01 -> 2021-04

CV results summary

In [29]:
print("Fold-by-fold results:")
cv_results_df

Fold-by-fold results:


,fold,train_start,train_end,valid_start,valid_end,n_train,n_valid,train_pos_rate,valid_pos_rate,roc_auc,pr_auc,top50_hit_rate,top50_positives,top100_hit_rate,top100_positives,top250_hit_rate,top250_positives,top500_hit_rate,top500_positives
0,1,2020-01,2020-06,2020-07,2020-07,14139,2648,0.102412,0.071375,0.817758,0.363766,0.58,29,0.46,46,0.348,87,0.232,116
1,2,2020-01,2020-07,2020-08,2020-08,16787,2648,0.097516,0.079683,0.824335,0.418743,0.72,36,0.58,58,0.380,95,0.264,132
2,3,2020-01,2020-08,2020-09,2020-09,19435,2648,0.095086,0.109517,0.793906,0.482695,0.82,41,0.63,63,0.524,131,0.346,173
3,4,2020-01,2020-09,2020-10,2020-10,22083,2648,0.096817,0.115559,0.794254,0.422916,0.76,38,0.60,60,0.472,118,0.334,167
4,5,2020-01,2020-10,2020-11,2020-11,24731,2648,0.098823,0.127266,0.795546,0.447401,0.74,37,0.68,68,0.492,123,0.368,184
5,6,2020-01,2020-11,2020-12,2020-12,27379,3170,0.101574,0.098107,0.802374,0.402974,0.60,30,0.60,60,0.444,111,0.362,181
6,7,2020-01,2020-12,2021-01,2021-01,30549,3407,0.101214,0.095098,0.829972,0.441833,0.70,35,0.61,61,0.500,125,0.390,195
7,8,2020-01,2021-01,2021-02,2021-02,33956,3406,0.100601,0.072519,0.843368,0.412772,0.66,33,0.59,59,0.396,99,0.298,149
8,9,2020-01,2021-02,2021-03,2021-03,37362,3406,0.098041,0.082501,0.880436,0.469827,0.76,38,0.58,58,0.472,118,0.356,178
9,10,2020-01,2021-03,2021-04,2021-04,40768,3406,0.096743,0.107164,0.817860,0.420684,0.68,34,0.62,62,0.524,131,0.400,200


In [31]:
print("Std metrics:")
cv_results_df[metric_cols].std(numeric_only=True).to_frame("std").T

Std metrics:


,n_train,n_valid,train_pos_rate,valid_pos_rate,roc_auc,pr_auc,top50_hit_rate,top50_positives,top100_hit_rate,top100_positives,top250_hit_rate,top250_positives,top500_hit_rate,top500_positives
std,58108.782535,1246.1848,0.008001,0.022608,0.026787,0.053561,0.101291,5.064566,0.092978,9.29779,0.067614,16.903446,0.053739,26.86928


In [33]:
cols_show = [c for c in ["EID", "MONTH", "PEAKID", "TARGET", "score", "cv_fold"] if c in oof_df.columns]

oof_ranked = oof_df.sort_values("score", ascending=False).reset_index(drop=True)

oof_ranked[cols_show].head(30)

,EID,MONTH,PEAKID,TARGET,score,cv_fold
0,5976,2021-04,0,0,0.986879,10
1,5976,2021-04,1,0,0.986679,10
2,5573,2021-06,0,1,0.986293,12
3,6585,2021-04,1,1,0.986218,10
4,5573,2021-09,1,1,0.985957,15
5,5573,2021-12,0,1,0.985657,18
6,5976,2021-09,0,1,0.985224,15
7,4588,2021-04,1,1,0.985120,10
8,5573,2021-07,1,1,0.985000,13
9,5573,2021-11,0,1,0.984872,17


In [35]:
feat_imp_summary = (
    feat_imp_df.groupby("feature", as_index=False)["importance"]
    .mean()
    .sort_values("importance", ascending=False)
)

feat_imp_summary.head(30)

,feature,importance
40,psm_activation_max,0.109872
18,pr_rolling3_mean,0.105614
22,profitable_count_3m,0.083509
21,profit_rolling3_mean,0.053252
14,pr_lag1,0.047136
34,psm_abs_max,0.046605
35,psm_abs_nonzero_mean,0.041229
39,psm_abs_sum,0.035475
19,profit_lag1,0.033925
10,is_sim_only,0.030574


In [37]:
X_full = df[feature_cols].fillna(0)
y_full = df["TARGET"].astype(int)

final_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=8,
    min_samples_leaf=20,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

final_model.fit(X_full, y_full)

print("Final model trained.")

Final model trained.
